# Giai đoạn 2 — Huấn luyện mô hình định vị

Đồ án tốt nghiệp Nhóm 15 · Khoa CNTT · Đại học Đà Lạt

So sánh bốn mô hình hồi quy toạ độ: **kNN**, **WKNN**, **Random Forest**, **XGBoost**.

Notebook gọi thẳng gói `ml/` trong kho mã, không chép lại logic — sửa ở kho là Colab
chạy theo.

---

### Chọn runtime: CPU, không cần GPU

Tập huấn luyện chỉ **547 mẫu × 36 đặc trưng ≈ 154 KB**. Ở quy mô này chi phí chuyển dữ
liệu sang GPU còn lớn hơn phép tính, nên GPU **chậm hơn** CPU. Để `Runtime → Change
runtime type → CPU`.

Lợi ích thật của Colab ở đây là môi trường đồng nhất cho cả nhóm và không chiếm máy cá
nhân trong lúc quét lưới tham số, không phải tốc độ.

| Việc | Thời gian đo trên máy để bàn |
|---|---|
| kNN + WKNN (20 tổ hợp) | vài giây |
| Random Forest (27 tổ hợp) | ~5 giây |
| XGBoost, lưới đầy đủ 324 tổ hợp | **~293 giây** |

Colab thường có 2 lõi nên chậm hơn khoảng 2–3 lần. Dùng `--nhanh` khi chỉ muốn thử.

---
## Phần 0 — Chuẩn bị

Chạy cả bốn ô theo thứ tự.

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/DoAnTotNghiep-Indoor/Indoor_Positoning_System-DATN.git"
DIR  = "Indoor_Positoning_System-DATN"

if os.path.isdir(DIR):
    subprocess.run(["git", "-C", DIR, "pull", "--quiet"], check=False)
    print("Đã cập nhật kho mã.")
else:
    subprocess.run(["git", "clone", "--quiet", REPO, DIR], check=True)
    print("Đã clone kho mã.")

os.chdir(DIR)
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

In [ ]:
# Colab có sẵn pandas, numpy, scikit-learn, joblib, matplotlib.
# xgboost thường cũng có sẵn — chỉ cài nếu thiếu.
import importlib.util, os, subprocess, sys

thieu = [m for m in ("pandas", "numpy", "sklearn", "joblib", "matplotlib", "xgboost")
         if importlib.util.find_spec(m) is None]

if thieu:
    ten_goi = {"sklearn": "scikit-learn"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[ten_goi.get(m, m) for m in thieu]], check=True)
    print("Đã cài:", thieu)
else:
    print("Đủ thư viện.")

import sklearn, xgboost
print(f"scikit-learn {sklearn.__version__} · xgboost {xgboost.__version__}")
print(f"Số lõi CPU: {os.cpu_count()}")

### Chạy lại pipeline giai đoạn 1

**Bắt buộc.** `data/splits/` không được commit lên kho (sinh lại được), nên phiên Colab
mới chưa có ba tệp train/validation/test. Ô này tạo ra chúng, mất dưới một giây.

In [ ]:
!python -m ml.pipeline

In [ ]:
# Kiểm tra đã đủ đầu vào cho huấn luyện chưa
from ml import config
import pandas as pd, json

fl = json.loads((config.ARTIFACTS_DIR / config.FEATURE_LIST_JSON).read_text(encoding="utf-8"))
print(f"Đặc trưng: {fl['feature_count']} cột AP")

for ten in ("train", "validation", "test"):
    d = pd.read_csv(config.SPLITS_DIR / f"{ten}.csv")
    print(f"  {ten:11s} X {d[fl['ap_columns']].shape}  y {d[['x','y']].shape}")

---
## Phần 1 — Huấn luyện

Quy trình cho cả bốn mô hình là một, để bảng so sánh có ý nghĩa:

1. Quét lưới tham số, chọn cấu hình tốt nhất trên tập **validation**
2. Huấn luyện lại cấu hình đó trên train + validation
3. Đánh giá **một lần** trên tập **test**

Bước 3 chỉ chạy đúng một lần. Chọn tham số dựa trên tập test rồi báo cáo kết quả trên
chính tập đó là tự lừa mình.

In [ ]:
# Lưới rút gọn — khoảng 30 giây, dùng để kiểm tra mọi thứ chạy được
!python -m ml.train --nhanh

In [ ]:
# Lưới đầy đủ theo tài liệu thiết kế — XGBoost quét 324 tổ hợp
# Trên Colab 2 lõi mất khoảng 10-15 phút. Chạy khi đã chắc chắn.
!python -m ml.train

---
## Phần 2 — Biểu đồ cho báo cáo

Bốn biểu đồ, lưu vào `reports/figures/`.

In [ ]:
import json, joblib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from ml import config, evaluate

MAU = {"kNN": "#8899aa", "WKNN": "#4a7fa5", "Random Forest": "#c98a3c", "XGBoost": "#0F6E8C"}

meta = json.loads((config.ARTIFACTS_DIR / "model_metadata.json").read_text(encoding="utf-8"))
bang = pd.read_csv(config.REPORTS_DIR / "tables" / "model_comparison.csv")
fl   = json.loads((config.ARTIFACTS_DIR / config.FEATURE_LIST_JSON).read_text(encoding="utf-8"))
ap   = fl["ap_columns"]

te = pd.read_csv(config.SPLITS_DIR / "test.csv")
X_te = te[ap].to_numpy(float)
y_te = te[["x", "y"]].to_numpy(float)

# Nạp lại từng mô hình và tính sai số từng mẫu
loi_theo_mo_hinh = {}
for khoa, thong_tin in meta["cac_mo_hinh"].items():
    model = joblib.load(config.ARTIFACTS_DIR / f"model_{khoa}.pkl")
    loi_theo_mo_hinh[thong_tin["ten"]] = evaluate.khoang_cach_loi(y_te, model.predict(X_te))

(config.REPORTS_DIR / "figures").mkdir(parents=True, exist_ok=True)
print("Đã nạp", len(loi_theo_mo_hinh), "mô hình. Mô hình active:", meta["mo_hinh_active"])

### Biểu đồ 1 — Đường CDF sai số

**Biểu đồ quan trọng nhất của cả đồ án.** Đọc theo chiều ngang: tại mốc 90%, mô hình nào
nằm bên trái hơn thì trường hợp xấu nhất của nó nhẹ hơn.

Chú ý đoạn dốc đứng ở sát 0 mét của kNN và WKNN — xem phần diễn giải bên dưới.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for ten, loi in loi_theo_mo_hinh.items():
    x, y = evaluate.duong_cdf(loi)
    ax.plot(x, y, label=ten, color=MAU.get(ten), lw=2)

for p in (50, 75, 90):
    ax.axhline(p, color="#999", lw=.6, ls=":", zorder=0)
    ax.text(ax.get_xlim()[1], p, f" {p}%", va="center", fontsize=8, color="#666")

ax.set_xlabel("Sai số khoảng cách (m)")
ax.set_ylabel("Tỉ lệ tích luỹ số mẫu (%)")
ax.set_title("CDF sai số định vị trên tập test")
ax.legend(loc="lower right")
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(config.REPORTS_DIR / "figures" / "cdf_error.png", dpi=150)
plt.show()

### Biểu đồ 2 — Phân bố sai số

Cho thấy rõ vì sao chỉ nhìn sai số trung bình là chưa đủ.

In [ ]:
fig, axes = plt.subplots(1, len(loi_theo_mo_hinh), figsize=(14, 3.4), sharey=True)

for ax, (ten, loi) in zip(np.atleast_1d(axes), loi_theo_mo_hinh.items()):
    ax.hist(loi, bins=np.arange(0, loi.max() + 3, 3), color=MAU.get(ten), edgecolor="white")
    ax.axvline(loi.mean(), color="#c0392b", lw=1.5, ls="--")
    ax.set_title(f"{ten}\ntrung bình {loi.mean():.1f} m", fontsize=10)
    ax.set_xlabel("Sai số (m)")
    ax.grid(alpha=.2, axis="y")

np.atleast_1d(axes)[0].set_ylabel("Số mẫu")
fig.suptitle("Phân bố sai số — đường đỏ là giá trị trung bình", y=1.04, fontsize=11)
fig.tight_layout()
fig.savefig(config.REPORTS_DIR / "figures" / "error_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Biểu đồ 3 — Bản đồ nhiệt sai số theo điểm tham chiếu

Chỗ nào sai nhiều thường là chỗ ít AP phủ tới hoặc bị che khuất. Đây là thông tin cụ thể
để cải thiện, khác hẳn một con số trung bình chung.

In [ ]:
theo_diem = pd.read_csv(config.REPORTS_DIR / "tables" / "error_by_reference_point.csv")

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(theo_diem["x"], theo_diem["y"], c=theo_diem["loi_trung_binh"],
                s=190, cmap="YlOrRd", edgecolor="#333", linewidth=.6)

for r in theo_diem.itertuples():
    ax.annotate(r.rp_id.replace("RP", ""), (r.x, r.y),
                fontsize=6.5, ha="center", va="center")

fig.colorbar(sc, ax=ax, label="Sai số trung bình (m)")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title(f"Sai số theo vị trí — mô hình {meta['mo_hinh_active']}")
ax.grid(alpha=.25)
ax.set_aspect("equal", adjustable="datalim")
fig.tight_layout()
fig.savefig(config.REPORTS_DIR / "figures" / "error_heatmap.png", dpi=150)
plt.show()

print("Năm điểm sai nhiều nhất:")
print(theo_diem.head(5)[["rp_id", "so_mau", "loi_trung_binh", "loi_lon_nhat"]].to_string(index=False))

### Biểu đồ 4 — Độ quan trọng đặc trưng của XGBoost

Trả lời câu hỏi hội đồng hay hỏi: AP nào thực sự đóng góp vào việc định vị. AP có độ
quan trọng gần 0 ở cả hai trục là ứng viên để siết ngưỡng lọc ở bước 5.

In [ ]:
from ml.models import xgboost_model

model_xgb = joblib.load(config.ARTIFACTS_DIR / "model_xgboost_model.pkl")
quan_trong = xgboost_model.do_quan_trong_dac_trung(model_xgb, ap)

dq = pd.DataFrame(quan_trong)
dq["tong"] = dq.sum(axis=1)
dq = dq.sort_values("tong", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
vt = np.arange(len(dq))
ax.barh(vt - .2, dq["x"], height=.38, label="trục x", color="#0F6E8C")
ax.barh(vt + .2, dq["y"], height=.38, label="trục y", color="#c98a3c")
ax.set_yticks(vt)
ax.set_yticklabels([b[-8:] for b in dq.index], fontsize=8, family="monospace")
ax.invert_yaxis()
ax.set_xlabel("Độ quan trọng")
ax.set_title("15 AP đóng góp nhiều nhất — XGBoost")
ax.legend()
ax.grid(alpha=.25, axis="x")
fig.tight_layout()
fig.savefig(config.REPORTS_DIR / "figures" / "feature_importance.png", dpi=150)
plt.show()

---
## Phần 3 — Đọc kết quả cho đúng

Chạy ô dưới trước khi viết bất kỳ dòng nào vào báo cáo.

In [ ]:
print("=" * 70)
print("SAI SỐ CÓ PHÂN BỐ HAI ĐỈNH — KHÔNG PHẢI PHÂN BỐ CHUẨN")
print("=" * 70)

tat = pd.read_csv(config.PROCESSED_DIR / "fingerprint_dataset_sorted.csv")
so_toa_do = tat[["x", "y"]].drop_duplicates().shape[0]
print(f"\n{len(tat)} mẫu huấn luyện nhưng chỉ có {so_toa_do} toạ độ khác nhau.")
print("Dữ liệu thu ĐÚNG TẠI các điểm tham chiếu, không có mẫu nào ở giữa hai điểm.")
print("Nhãn (x, y) vì thế bị lượng tử hoá theo lưới ~7 m.\n")

for ten, loi in loi_theo_mo_hinh.items():
    dung_khop = (loi < 1e-9).mean() * 100
    khi_sai = loi[loi > 1e-9].mean() if (loi > 1e-9).any() else 0
    print(f"  {ten:15s} đúng khít {dung_khop:5.1f}% số mẫu · khi sai thì lệch {khi_sai:5.2f} m")

print("\nkNN/WKNN chọn đúng điểm thì sai số bằng 0 tuyệt đối, chọn sai thì lệch cả chục")
print("mét. Không mẫu nào có sai số gần giá trị trung bình — nên con số trung bình KHÔNG")
print("mô tả được mẫu điển hình. Phải báo cáo kèm CDF và tỉ lệ đúng khít.")

In [ ]:
print("=" * 70)
print("MÔ HÌNH NÀO TỐT HƠN? TUỲ CHỈ SỐ.")
print("=" * 70)
print()
print(bang[["mo_hinh", "loi_trung_binh", "cdf_50", "cdf_75", "cdf_90",
            "loi_lon_nhat", "thoi_gian_du_doan_ms"]]
      .to_string(index=False, float_format=lambda v: f"{v:7.2f}"))
print()

tot_tb  = bang.loc[bang["loi_trung_binh"].idxmin(), "mo_hinh"]
tot_90  = bang.loc[bang["cdf_90"].idxmin(), "mo_hinh"]
tot_max = bang.loc[bang["loi_lon_nhat"].idxmin(), "mo_hinh"]

print(f"  Sai số trung bình thấp nhất : {tot_tb}")
print(f"  CDF90 thấp nhất             : {tot_90}")
print(f"  Sai số lớn nhất nhỏ nhất    : {tot_max}")
print()
print("Với hệ thống định vị, trường hợp xấu nhất quan trọng hơn giá trị trung bình:")
print("marker nhảy 50 m một lần tệ hơn nhiều so với lệch đều 8 m.")

---
## Phần 4 — Tải kết quả về

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("ips_models",  "zip", ".", "artifacts")
shutil.make_archive("ips_reports", "zip", ".", "reports")

for ten in ("ips_models.zip", "ips_reports.zip"):
    print(f"{os.path.getsize(ten)/1024:8.1f} KB  {ten}")
    files.download(ten)

`ips_models.zip` gồm bốn tệp `model_*.pkl`, `model_metadata.json`, `scaler.pkl` và
`feature_list.json`. Ba tệp cuối là **hợp đồng dữ liệu** mà backend cần ở giai đoạn 3 —
phải đi cùng nhau, sinh ra từ cùng một lần chạy, không được trộn từ các lần khác nhau.

`ips_reports.zip` gồm bốn biểu đồ và ba bảng CSV để chèn vào báo cáo.